## MÔN HỌC: Kỹ thuật xây dựng hệ thống Agentic AI - SE373.R11
### BTVN #2: Xây dựng ISSUE TRIAGE MINI-APP
### Người thực hiện: Nguyễn Hùng Cường (23520201)


---
## 1. Khởi tạo môi trường & Cấu hình OpenRouter / OpenAI API

Chúng ta nạp các biến cấu hình từ file `.env`:
- `OPENAI_API_KEY`: API Key lấy từ [OpenRouter](https://openrouter.ai/keys).
- `OPENAI_BASE_URL`: Endpoint OpenAI-compatible (`https://openrouter.ai/api/v1`).
- `OPENAI_MODEL`: Sử dụng mô hình LLM free tier `nex-agi/nex-n2.5-pro:free`.


In [1]:
import os
import sys
import json
from typing import Literal, Any, Dict, List, Optional
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from dotenv import load_dotenv
from openai import OpenAI

# Nạp file .env từ thư mục hiện tại hoặc thư mục cha
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
MODEL = os.getenv("OPENAI_MODEL", "google/gemma-4-26b-a4b-it:free")

if not API_KEY:
    raise ValueError("Chưa tìm thấy OPENAI_API_KEY trong file .env! Vui lòng cấu hình trước khi chạy.")

print(f"✅ Đã tải cấu hình thành công!")
print(f" - Provider Base URL: {BASE_URL}")
print(f" - Model được chọn:   {MODEL}")

# Khởi tạo client tương thích chuẩn OpenAI
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


✅ Đã tải cấu hình thành công!
 - Provider Base URL: https://openrouter.ai/api/v1
 - Model được chọn:   nex-agi/nex-n2.5-pro:free


---
## 2. Định nghĩa Schema dữ liệu `IssueTriage` bằng Pydantic

Schema tối thiểu của `IssueTriage` gồm có:
- `status`: Bắt buộc là một trong 3 giá trị: `"classified"` | `"insufficient_data"` | `"out_of_scope"`.
- `severity`: `"P0"` | `"P1"` | `"P2"` | `"P3"` | `None`.
  - **P0**: Lỗi tê liệt hệ thống, thanh toán dừng toàn bộ, sự cố khẩn cấp mức cao nhất.
  - **P1**: Lỗi nghiêm trọng ảnh hưởng diện rộng, chức năng cốt lõi bị lỗi.
  - **P2**: Lỗi chức năng thứ yếu, hệ thống chậm hoặc chỉ ảnh hưởng nhóm nhỏ.
  - **P3**: Lỗi giao diện nhỏ, typo, không ảnh hưởng logic.
- `component`: Tên thành phần gặp sự cố (ví dụ: `payment`, `identity`, `search`, ...).
- `needs_urgent_response`: `bool` (cần can thiệp khẩn cấp ngay hay không).
- `reason`: Lý do ngắn gọn dựa trên dữ liệu issue.

> **Ràng buộc:** Cấu hình `model_config = ConfigDict(extra="forbid")` để ngăn model tự chế thêm các trường không mong muốn.


In [2]:
class IssueTriage(BaseModel):
    """Schema hợp đồng dữ liệu giữa Model và Application."""
    model_config = ConfigDict(extra="forbid")

    status: Literal["classified", "insufficient_data", "out_of_scope"] = Field(
        description="Trạng thái phân loại: classified (đã phân loại), insufficient_data (thiếu thông tin), out_of_scope (ngoài phạm vi sự cố)"
    )
    severity: Literal["P0", "P1", "P2", "P3"] | None = Field(
        default=None,
        description="Mức độ nghiêm trọng từ P0 đến P3. Nếu không phân loại được thì để None."
    )
    component: Optional[str] = Field(
        default=None,
        description="Component phần mềm liên quan (ví dụ: payment, identity, search...)"
    )
    needs_urgent_response: bool = Field(
        default=False,
        description="Cờ đánh dấu có cần can thiệp khẩn cấp (on-call/hotfix) ngay lập tức hay không."
    )
    reason: str = Field(
        description="Lý do ngắn gọn giải thích cho quyết định phân loại."
    )

print("✅ Đã khởi tạo Pydantic Schema IssueTriage:")
print(json.dumps(IssueTriage.model_json_schema(), ensure_ascii=False, indent=2))


✅ Đã khởi tạo Pydantic Schema IssueTriage:
{
  "additionalProperties": false,
  "description": "Schema hợp đồng dữ liệu giữa Model và Application.",
  "properties": {
    "status": {
      "description": "Trạng thái phân loại: classified (đã phân loại), insufficient_data (thiếu thông tin), out_of_scope (ngoài phạm vi sự cố)",
      "enum": [
        "classified",
        "insufficient_data",
        "out_of_scope"
      ],
      "title": "Status",
      "type": "string"
    },
    "severity": {
      "anyOf": [
        {
          "enum": [
            "P0",
            "P1",
            "P2",
            "P3"
          ],
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Mức độ nghiêm trọng từ P0 đến P3. Nếu không phân loại được thì để None.",
      "title": "Severity"
    },
    "component": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
      

---
## 3. Thiết kế Prompt Template: Tách biệt Instruction & Input

Theo nguyên tắc của kỹ thuật thiết kế Prompt cho LLM:
- **System Instruction**: Xác định Persona (chuyên gia Triage sự cố), định nghĩa rõ ràng các mức `P0-P3`, quy định điều kiện gọi Tool và định dạng phản hồi.
- **User Prompt (Input)**: Chứa dữ liệu issue cụ thể cần xử lý + định dạng JSON Schema mong muốn.


In [3]:
SYSTEM_INSTRUCTION = """Bạn là kỹ sư phụ trách phân loại sự cố phần mềm (Issue Triage Engineer).

Nhiệm vụ của bạn:
1. Phân loại mức độ nghiêm trọng (severity) của issue:
   - P0: Toàn bộ hệ thống ngừng hoạt động, chức năng thanh toán tê liệt hoàn toàn, ảnh hưởng tất cả khách hàng.
   - P1: Sự cố nghiêm trọng ở chức năng chính, ảnh hưởng diện rộng, cần xử lý trong ngày.
   - P2: Sự cố chức năng phụ, hiệu năng suy giảm, có giải pháp thay thế tạm thời.
   - P3: Lỗi nhỏ về giao diện, hiển thị, không ảnh hưởng nghiệp vụ cốt lõi.
2. Xử lý trường hợp đặc biệt:
   - Nếu mô tả quá ngắn hoặc không đủ thông tin kết luận: đặt status="insufficient_data", severity=null.
   - Nếu nội dung không phải lỗi phần mềm (ví dụ: hỏi thời tiết, chào hỏi, spam): đặt status="out_of_scope", severity=null.
3. Sử dụng Tool (Function Calling):
   - Khi cần biết team phụ trách component nào, HÃY GỌI TOOL `get_component_owner`.
   - Không tự đoán tên team phụ trách nếu chưa gọi tool.
4. Đầu ra:
   - Sau khi có kết quả tool (nếu có), hãy phản hồi kết quả phân loại cuối cùng dưới dạng JSON đúng theo schema IssueTriage được cung cấp.
   - Tuyệt đối không thêm text bên ngoài đối tượng JSON.
"""

def build_user_prompt(issue_description: str) -> str:
    """Prompt template kết hợp Input mô tả issue và Schema ràng buộc."""
    schema_spec = json.dumps(IssueTriage.model_json_schema(), ensure_ascii=False, indent=2)
    return f"""=== MÔ TẢ ISSUE ĐẦU VÀO ===
{issue_description}

=== YÊU CẦU ĐỊNH DẠNG ĐẦU RA ===
Hãy phân tích và trả về DUY NHẤT một JSON object tuân thủ schema Pydantic sau:
{schema_spec}
"""

# Thử nghiệm tạo prompt cho một issue mẫu
sample_issue = "Nút thanh toán trả HTTP 500 với mọi thẻ Visa từ 14:30. Hãy triage issue và cho biết team nào cần xử lý."
print("=== DEMO PROMPT ĐƯỢC TẠO ===")
print(build_user_prompt(sample_issue))


=== DEMO PROMPT ĐƯỢC TẠO ===
=== MÔ TẢ ISSUE ĐẦU VÀO ===
Nút thanh toán trả HTTP 500 với mọi thẻ Visa từ 14:30. Hãy triage issue và cho biết team nào cần xử lý.

=== YÊU CẦU ĐỊNH DẠNG ĐẦU RA ===
Hãy phân tích và trả về DUY NHẤT một JSON object tuân thủ schema Pydantic sau:
{
  "additionalProperties": false,
  "description": "Schema hợp đồng dữ liệu giữa Model và Application.",
  "properties": {
    "status": {
      "description": "Trạng thái phân loại: classified (đã phân loại), insufficient_data (thiếu thông tin), out_of_scope (ngoài phạm vi sự cố)",
      "enum": [
        "classified",
        "insufficient_data",
        "out_of_scope"
      ],
      "title": "Status",
      "type": "string"
    },
    "severity": {
      "anyOf": [
        {
          "enum": [
            "P0",
            "P1",
            "P2",
            "P3"
          ],
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "M

---
## 4. Khai báo Tool & Cơ chế Application-Controlled Execution

Chúng ta định nghĩa tool `get_component_owner`:
- Input: `component` (string)
- Output: Tên team chịu trách nhiệm (`owner`)


In [4]:
# Cơ sở dữ liệu danh mục phụ trách nội bộ của Application
COMPONENT_OWNERS = {
    "payment": "checkout-platform",
    "identity": "identity-platform",
    "search": "search-platform",
    "infrastructure": "infra-core",
    "database": "data-infra"
}

# Khai báo Tool Schema theo chuẩn OpenAI Function Calling
FUNCTION_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_component_owner",
            "description": "Lấy thông tin team chịu trách nhiệm (owner) cho một component phần mềm. Chỉ dùng các component hợp lệ trong schema.",
            "parameters": {
                "type": "object",
                "properties": {
                    "component": {
                        "type": "string",
                        "enum": list(COMPONENT_OWNERS.keys()),
                        "description": "Tên component cần tra cứu owner (payment, identity, search, infrastructure, database)"
                    }
                },
                "required": ["component"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
]

def execute_get_component_owner(component: str) -> str:
    """Hàm thực thi an toàn tại Application."""
    comp = component.strip().lower()
    return COMPONENT_OWNERS.get(comp, "general-tier1-support")

print("✅ Đã khai báo Tool get_component_owner:")
print(json.dumps(FUNCTION_TOOLS, ensure_ascii=False, indent=2))


✅ Đã khai báo Tool get_component_owner:
[
  {
    "type": "function",
    "function": {
      "name": "get_component_owner",
      "description": "Lấy thông tin team chịu trách nhiệm (owner) cho một component phần mềm. Chỉ dùng các component hợp lệ trong schema.",
      "parameters": {
        "type": "object",
        "properties": {
          "component": {
            "type": "string",
            "enum": [
              "payment",
              "identity",
              "search",
              "infrastructure",
              "database"
            ],
            "description": "Tên component cần tra cứu owner (payment, identity, search, infrastructure, database)"
          }
        },
        "required": [
          "component"
        ],
        "additionalProperties": false
      },
      "strict": true
    }
  }
]


---
## 5. Application-Side Validation (Pydantic + Business Rules)

Hàm `validate_triage_result` thực hiện 2 tầng kiểm tra chặt chẽ:
1. **Schema Validation (Pydantic)**: Sử dụng `IssueTriage.model_validate_json(raw_json)` để parse và validate từng field, kiểu dữ liệu, các enum literal.
2. **Business Logic Validation**:
   - Nếu `status == 'classified'`: Bắt buộc `severity` không được là `None`.
   - Nếu `status in ['insufficient_data', 'out_of_scope']`: Bắt buộc `severity` phải là `None`.
   - Nếu `needs_urgent_response == True`: Mức độ `severity` chỉ nên là `P0` hoặc `P1`.


In [5]:
def validate_triage_result(raw_json_str: str) -> IssueTriage:
    """
    Validate output từ model ở phía Application:
    1. Parse và validate bằng Pydantic model_validate_json (KHÔNG dùng regex/substring).
    2. Kiểm tra các quy tắc nghiệp vụ (Business Rules).
    """
    # Bước 1: Validate Schema với Pydantic
    try:
        data = IssueTriage.model_validate_json(raw_json_str)
    except ValidationError as err:
        raise ValueError(f"❌ Schema validation thất bại: {err}") from err

    # Bước 2: Validate Business Logic
    if data.status == "classified":
        if data.severity is None:
            raise ValueError("❌ Vi phạm nghiệp vụ: Khi status='classified', trường severity không được để trống (None)!")
    elif data.status in ["insufficient_data", "out_of_scope"]:
        if data.severity is not None:
            raise ValueError(f"❌ Vi phạm nghiệp vụ: Khi status='{data.status}', trường severity bắt buộc phải là None, nhưng nhận được '{data.severity}'!")
        if data.needs_urgent_response:
            print(f"⚠️ Cảnh báo nghiệp vụ: status='{data.status}' nhưng needs_urgent_response lại là True!")

    return data

print("✅ Đã khởi tạo hàm validate_triage_result thành công.")


✅ Đã khởi tạo hàm validate_triage_result thành công.


---
## 6. Điều phối Quy trình Triage & In Trace Tuần Tự

Chu trình hoàn chỉnh in ra đúng 4 bước trace theo yêu cầu:
$$\text{1. tool\_call} \longrightarrow \text{2. application executes} \longrightarrow \text{3. tool\_result} \longrightarrow \text{4. final response}$$

Kèm theo việc đo lường số lượng Token tiêu thụ (Prompt tokens, Completion tokens, Total tokens) trong mỗi round gọi LLM.


In [6]:
def triage_issue_workflow(issue_text: str, tool_choice: str = "auto", verbose: bool = True) -> tuple[Optional[IssueTriage], dict]:
    """
    Điều phối luồng Issue Triage:
    - Gửi request kèm tool schema (tool_choice='auto' hoặc 'required')
    - Xử lý tool call nếu model yêu cầu
    - Trả kết quả tool về model
    - Nhận phản hồi cuối cùng và validate
    - In trace trực quan chi tiết
    """
    if verbose:
        print("=" * 80)
        print("📌 BẮT ĐẦU XỬ LÝ ISSUE:")
        print(f"   {repr(issue_text)}")
        print("=" * 80)

    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": build_user_prompt(issue_text)}
    ]

    total_prompt_tokens = 0
    total_completion_tokens = 0

    # ROUND 1: Gửi Prompt và Tools đến Model
    if verbose:
        print(f"\n🚀 [ROUND 1] Gửi request đến Model (tools enabled, tool_choice={repr(tool_choice)})...")

    response_1 = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=FUNCTION_TOOLS,
        tool_choice=tool_choice
    )

    choice_1 = response_1.choices[0]
    msg_1 = choice_1.message
    usage_1 = response_1.usage

    if usage_1:
        total_prompt_tokens += usage_1.prompt_tokens
        total_completion_tokens += usage_1.completion_tokens
        if verbose:
            print(f"   📊 Token Round 1: {usage_1.prompt_tokens} input + {usage_1.completion_tokens} output = {usage_1.total_tokens} total")

    final_raw_content = ""

    # Kiểm tra xem Model có đề xuất gọi Tool hay không
    if msg_1.tool_calls:
        messages.append(msg_1)
        for tc in msg_1.tool_calls:
            if verbose:
                print("\n" + "─" * 60)
                print("👉 [TRACE 1/4] Model đề xuất tool call (tool_call):")
                print(f"   - Tool Call ID: {tc.id}")
                print(f"   - Tên function: {tc.function.name}")
                print(f"   - Tham số:      {tc.function.arguments}")

            # [TRACE 2] Application kiểm soát và thực thi
            args = json.loads(tc.function.arguments)
            component_val = args.get("component", "")
            owner_val = execute_get_component_owner(component_val)
            tool_res_dict = {"component": component_val, "owner": owner_val}

            if verbose:
                print("\n⚙️ [TRACE 2/4] Application kiểm soát & thực thi (application executes):")
                print(f"   - Hàm thực thi: execute_get_component_owner({repr(component_val)})")
                print(f"   - Kết quả trả về: {repr(owner_val)}")

            # [TRACE 3] Gửi tool result quay lại model
            if verbose:
                print("\n🔄 [TRACE 3/4] Gửi kết quả tool quay lại model (tool_result):")
                print(f"   - Payload gửi lại: {json.dumps(tool_res_dict, ensure_ascii=False)}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_res_dict, ensure_ascii=False)
            })

        # ROUND 2: Model tổng hợp và trả về kết quả có cấu trúc
        if verbose:
            print("\n🚀 [ROUND 2] Gửi ngữ cảnh đầy đủ để Model trả về kết quả phân loại cuối cùng...")

        response_2 = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            response_format={"type": "json_object"}
        )

        choice_2 = response_2.choices[0]
        final_raw_content = choice_2.message.content or ""
        usage_2 = response_2.usage

        if usage_2:
            total_prompt_tokens += usage_2.prompt_tokens
            total_completion_tokens += usage_2.completion_tokens
            if verbose:
                print(f"   📊 Token Round 2: {usage_2.prompt_tokens} input + {usage_2.completion_tokens} output = {usage_2.total_tokens} total")

    else:
        if verbose:
            print("\nℹ️ Model không yêu cầu gọi tool trong trường hợp này.")
        final_raw_content = msg_1.content or ""

    # [TRACE 4] In Final Response từ Model
    if verbose:
        print("\n" + "─" * 60)
        print("🏁 [TRACE 4/4] Final Response từ Model (JSON thô):")
        print(final_raw_content)

    # Application-side Validation
    if verbose:
        print("\n🔍 TIẾN HÀNH VALIDATE Ở TẦNG APPLICATION:")

    validated_result = None
    try:
        validated_result = validate_triage_result(final_raw_content)
        if verbose:
            print("✅ VALIDATION THÀNH CÔNG! Dữ liệu đạt chuẩn IssueTriage:")
            print(json.dumps(validated_result.model_dump(), ensure_ascii=False, indent=2))
    except Exception as exc:
        if verbose:
            print(f"❌ LỖI VALIDATION: {exc}")

    metrics = {
        "prompt_tokens": total_prompt_tokens,
        "completion_tokens": total_completion_tokens,
        "total_tokens": total_prompt_tokens + total_completion_tokens
    }

    if verbose:
        print("\n" + "=" * 80)
        print(f"📈 TỔNG KẾT TOKEN: {metrics['prompt_tokens']} prompt + {metrics['completion_tokens']} completion = {metrics['total_tokens']} tổng token")
        print("=" * 80 + "\n")

    return validated_result, metrics

print("✅ Đã định nghĩa hàm điều phối workflow triage_issue_workflow.")


✅ Đã định nghĩa hàm điều phối workflow triage_issue_workflow.


---
## 7. Kiểm thử các trường hợp thực tế (Test Cases)

### Test Case 1: Sự cố thanh toán nghiêm trọng (Yêu cầu gọi tool & xếp hạng P1/P0)
- **Mô tả:** *"Nút thanh toán trả HTTP 500 với mọi thẻ Visa từ 14:30. Hãy triage issue và cho biết team nào cần xử lý."*
- **Quy trình kỳ vọng:**
  1. `tool_call`: Model nhận diện yêu cầu tra cứu team phụ trách và phát ra yêu cầu gọi `get_component_owner(component="payment")`.
  2. `application executes`: Ứng dụng thực thi hàm an toàn và tìm ra `checkout-platform`.
  3. `tool_result`: Ứng dụng đưa kết quả trả lại vào luồng hội thoại của Model.
  4. `final response`: Model tổng hợp thông tin, trả về JSON `IssueTriage` với `status="classified"`, `severity="P0"` hoặc `"P1"`, `needs_urgent_response=True`.
  5. `application validation`: Pydantic & Business Logic xác nhận hợp lệ 100%.


In [7]:
issue_tc1 = "Nút thanh toán trả HTTP 500 với mọi thẻ Visa từ 14:30. Hãy triage issue và cho biết team nào cần xử lý."
result_tc1, metrics_tc1 = triage_issue_workflow(issue_tc1, tool_choice="required")


📌 BẮT ĐẦU XỬ LÝ ISSUE:
   'Nút thanh toán trả HTTP 500 với mọi thẻ Visa từ 14:30. Hãy triage issue và cho biết team nào cần xử lý.'

🚀 [ROUND 1] Gửi request đến Model (tools enabled, tool_choice='required')...
   📊 Token Round 1: 1220 input + 140 output = 1360 total

────────────────────────────────────────────────────────────
👉 [TRACE 1/4] Model đề xuất tool call (tool_call):
   - Tool Call ID: call_f08599a015954ad3abfd8b90
   - Tên function: get_component_owner
   - Tham số:      {"component": "payment"}

⚙️ [TRACE 2/4] Application kiểm soát & thực thi (application executes):
   - Hàm thực thi: execute_get_component_owner('payment')
   - Kết quả trả về: 'checkout-platform'

🔄 [TRACE 3/4] Gửi kết quả tool quay lại model (tool_result):
   - Payload gửi lại: {"component": "payment", "owner": "checkout-platform"}

🚀 [ROUND 2] Gửi ngữ cảnh đầy đủ để Model trả về kết quả phân loại cuối cùng...
   📊 Token Round 2: 1075 input + 136 output = 1211 total

───────────────────────────────────────

---
### 🧪 Test Case 2: Dữ liệu không đầy đủ (`insufficient_data`)
- **Mô tả:** *"Hệ thống bị chậm lúc trưa nay."*
- **Kỳ vọng:**
  1. Model nhận thấy không đủ chi tiết kỹ thuật để kết luận severity hay component.
  2. Trả về `status="insufficient_data"`, `severity=None`.
  3. Application validate thành công quy tắc: khi `insufficient_data`, `severity` phải là `None`.


In [8]:
issue_tc2 = "Hệ thống bị chậm lúc trưa nay."
result_tc2, metrics_tc2 = triage_issue_workflow(issue_tc2, tool_choice="auto")


📌 BẮT ĐẦU XỬ LÝ ISSUE:
   'Hệ thống bị chậm lúc trưa nay.'

🚀 [ROUND 1] Gửi request đến Model (tools enabled, tool_choice='auto')...
   📊 Token Round 1: 1193 input + 178 output = 1371 total

ℹ️ Model không yêu cầu gọi tool trong trường hợp này.

────────────────────────────────────────────────────────────
🏁 [TRACE 4/4] Final Response từ Model (JSON thô):
{"status":"insufficient_data","severity":null,"component":null,"needs_urgent_response":false,"reason":"Mô tả quá ngắn, chưa rõ phạm vi ảnh hưởng, mức độ suy giảm hiệu năng, thời gian kéo dài và chức năng bị tác động."}

🔍 TIẾN HÀNH VALIDATE Ở TẦNG APPLICATION:
✅ VALIDATION THÀNH CÔNG! Dữ liệu đạt chuẩn IssueTriage:
{
  "status": "insufficient_data",
  "severity": null,
  "component": null,
  "needs_urgent_response": false,
  "reason": "Mô tả quá ngắn, chưa rõ phạm vi ảnh hưởng, mức độ suy giảm hiệu năng, thời gian kéo dài và chức năng bị tác động."
}

📈 TỔNG KẾT TOKEN: 1193 prompt + 178 completion = 1371 tổng token



---
### 🧪 Test Case 3: Yêu cầu ngoài phạm vi (`out_of_scope`)
- **Mô tả:** *"Cho mình hỏi thời tiết TP. Hồ Chí Minh hôm nay nắng hay mưa vậy bạn?"*
- **Kỳ vọng:**
  1. Model nhận diện đây không phải là báo cáo sự cố phần mềm.
  2. Trả về `status="out_of_scope"`, `severity=None`.
  3. Application validate thành công.


In [9]:
issue_tc3 = "Cho mình hỏi thời tiết TP. Hồ Chí Minh hôm nay nắng hay mưa vậy bạn?"
result_tc3, metrics_tc3 = triage_issue_workflow(issue_tc3)


📌 BẮT ĐẦU XỬ LÝ ISSUE:
   'Cho mình hỏi thời tiết TP. Hồ Chí Minh hôm nay nắng hay mưa vậy bạn?'

🚀 [ROUND 1] Gửi request đến Model (tools enabled, tool_choice='auto')...
   📊 Token Round 1: 1203 input + 115 output = 1318 total

ℹ️ Model không yêu cầu gọi tool trong trường hợp này.

────────────────────────────────────────────────────────────
🏁 [TRACE 4/4] Final Response từ Model (JSON thô):
{"status":"out_of_scope","severity":null,"component":null,"needs_urgent_response":false,"reason":"Đây là câu hỏi thời tiết, không phải lỗi phần mềm."}

🔍 TIẾN HÀNH VALIDATE Ở TẦNG APPLICATION:
✅ VALIDATION THÀNH CÔNG! Dữ liệu đạt chuẩn IssueTriage:
{
  "status": "out_of_scope",
  "severity": null,
  "component": null,
  "needs_urgent_response": false,
  "reason": "Đây là câu hỏi thời tiết, không phải lỗi phần mềm."
}

📈 TỔNG KẾT TOKEN: 1203 prompt + 115 completion = 1318 tổng token



---
## 8. Đo lường Token & Báo cáo Ước tính Chi phí (`docs/uoc_tinh_chi_phi.html`)

Đoạn mã bên dưới tính toán bảng tổng hợp từ số liệu token thực tế của **Test Case 1**:


In [10]:
input_tokens = metrics_tc1["prompt_tokens"]
output_tokens = metrics_tc1["completion_tokens"]
total_tokens = metrics_tc1["total_tokens"]

MONTHLY_ISSUES = 10_000
RATE_VND = 25_400

total_monthly_input = (MONTHLY_ISSUES * input_tokens) / 1_000_000
total_monthly_output = (MONTHLY_ISSUES * output_tokens) / 1_000_000

# Đơn giá các mô hình (/1M tokens)
models_pricing = [
    {
        "name": "OpenRouter Free Tier (nex-agi/gemma)",
        "in_price": 0.00,
        "out_price": 0.00,
        "note": "Phù hợp học tập / BTVN"
    },
    {
        "name": "OpenAI GPT-4o-mini",
        "in_price": 0.150,
        "out_price": 0.600,
        "note": "Khuyên dùng cho Production (Nhanh & rẻ)"
    },
    {
        "name": "DeepSeek V3",
        "in_price": 0.140,
        "out_price": 0.280,
        "note": "Chi phí rẻ nhất"
    },
    {
        "name": "Claude 3.5 Haiku",
        "in_price": 0.800,
        "out_price": 4.000,
        "note": "Độ chính xác cao cấp"
    }
]

print("=" * 85)
print(f"📊 BẢNG ƯỚC TÍNH CHI PHÍ VẬN HÀNH CHO {MONTHLY_ISSUES:,} ISSUES / THÁNG:")
print(f" - Input Token trung bình / issue:  {input_tokens} tokens  --> {total_monthly_input:.2f}M tokens/tháng")
print(f" - Output Token trung bình / issue: {output_tokens} tokens  --> {total_monthly_output:.2f}M tokens/tháng")
print(f" - Tổng Token / 1 issue:            {total_tokens} tokens")
print("=" * 85)
print(f"{'Mô hình':<35} | {'Chi phí ($)':<12} | {'Chi phí (VNĐ)':<15} | {'Ghi chú'}")
print("-" * 85)

for p in models_pricing:
    cost_usd = (total_monthly_input * p["in_price"]) + (total_monthly_output * p["out_price"])
    cost_vnd = cost_usd * RATE_VND
    print(f"{p['name']:<35} | ${cost_usd:<11.2f} | {int(cost_vnd):>10,d} VNĐ | {p['note']}")

print("=" * 85)
print("📄 Xem báo cáo chi tiết, biểu đồ trực quan và Simulator tương tác tại:")
print("   👉 docs/uoc_tinh_chi_phi.html")


📊 BẢNG ƯỚC TÍNH CHI PHÍ VẬN HÀNH CHO 10,000 ISSUES / THÁNG:
 - Input Token trung bình / issue:  2295 tokens  --> 22.95M tokens/tháng
 - Output Token trung bình / issue: 276 tokens  --> 2.76M tokens/tháng
 - Tổng Token / 1 issue:            2571 tokens
Mô hình                             | Chi phí ($)  | Chi phí (VNĐ)   | Ghi chú
-------------------------------------------------------------------------------------
OpenRouter Free Tier (nex-agi/gemma) | $0.00        |          0 VNĐ | Phù hợp học tập / BTVN
OpenAI GPT-4o-mini                  | $5.10        |    129,501 VNĐ | Khuyên dùng cho Production (Nhanh & rẻ)
DeepSeek V3                         | $3.99        |    101,239 VNĐ | Chi phí rẻ nhất
Claude 3.5 Haiku                    | $29.40       |    746,760 VNĐ | Độ chính xác cao cấp
📄 Xem báo cáo chi tiết, biểu đồ trực quan và Simulator tương tác tại:
   👉 docs/uoc_tinh_chi_phi.html
